# 1장 요약통계, 분포, 검정의 기본 (R과 파이썬으로 하는 통계분석)

보건통계학 기초 0–2장의 예로 요약통계와 결측, 정규분포·t 분포의 확률과 경계값, 요약값만으로 하는 t 검정, 같은 표의 χ²와 z, 다중비교 보정, 부트스트랩, 분위수·왜도·정규성 확인, 순위합 검정의 세부를 R로 계산합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요. 패키지가 없으면 먼저 install.packages(c("dplyr", "boot", "e1071", "nortest", "coin", "brunnermunzel"))를 실행합니다.


## a. 환자 단위로 모으고 요약하기

### 셀 1. 처방 단위 자료를 환자 단위로 모으기

In [ ]:
library(dplyr, warn.conflicts = FALSE)
rx <- data.frame(
  patient = c("A", "A", "A", "B", "C", "C"),
  date    = as.Date(c("2024-01-03", "2024-02-01", "2024-03-04",
                      "2024-01-10", "2024-01-15", "2024-03-20")),
  days    = c(30, 30, 30, 90, 30, 30))       # 공급일수
rx |>
  group_by(patient) |>
  summarise(first_date = min(date), n_rx = n(), total_days = sum(days))

### 셀 2. 평균, 분산, 표준편차

In [ ]:
x <- c(8, 12, 9, 11, 10)                  # A 병동 5명의 재원일수(일)
c(mean = mean(x), var = var(x), sd = sd(x))      # var와 sd는 n − 1로 나눔
sqrt(sum((x - mean(x))^2) / length(x))           # n으로 나누면

### 셀 3. 값이 빈 환자가 있으면

In [ ]:
x_na <- c(8, 12, NA, 11, 10)              # 셋째 환자의 값이 빔
mean(x_na)                                # NA가 하나라도 있으면 NA
c(mean = mean(x_na, na.rm = TRUE), sd = sd(x_na, na.rm = TRUE),
  n = sum(!is.na(x_na)))
cost <- c(0, 0, 12, 18, 20)               # 1인당 비용(만 원)
c(all = mean(cost), drop_zero = mean(cost[cost > 0]))

## b. 분포에서 확률과 경계값 구하기

### 셀 4. 정규분포의 확률과 경계값

In [ ]:
pnorm(-1.5)                        # P(Z < −1.5)
pnorm(2.0, lower.tail = FALSE)     # P(Z > 2.0)
qnorm(0.975)                       # 위쪽 2.5%를 자르는 z

### 셀 5. 표준오차와 t 경계값

In [ ]:
se1 <- 7.0 / sqrt(30)              # 신약군 평균의 SE
se2 <- 7.2 / sqrt(30)              # 위약군 평균의 SE
se_diff <- sqrt(se1^2 + se2^2)     # 차이의 SE
round(c(se1, se2, se_diff), 2)
round(qt(0.975, df = c(5, 29, 58, 1000)), 2)   # 자유도별 t 경계값

### 셀 6. 통계량에서 p, 확률에서 임계값

In [ ]:
qt(0.975, 58)                          # t(58)의 양측 5% 임계값
2 * pt(2.35, 58, lower.tail = FALSE)   # t(58) = 2.35의 양측 p
2 * pnorm(1.96, lower.tail = FALSE)    # z = 1.96의 양측 p
pnorm(0.25)                            # Φ(0.25): 검정력 계산
pchisq(5.31, 1, lower.tail = FALSE)    # χ²(1) = 5.31의 p
pf(4.12, 2, 87, lower.tail = FALSE)    # F(2, 87) = 4.12의 p

## c. 요약값으로 하는 t 검정

### 셀 7. 평균, SD, 인원만으로 t 검정

In [ ]:
t_from_stats <- function(m1, s1, n1, m2, s2, n2, var.equal = FALSE) {
  if (var.equal) {                       # Student: 합동분산
    sp2 <- ((n1 - 1) * s1^2 + (n2 - 1) * s2^2) / (n1 + n2 - 2)
    se <- sqrt(sp2 * (1 / n1 + 1 / n2)); df <- n1 + n2 - 2
  } else {                               # Welch: 분산을 군마다 따로
    v1 <- s1^2 / n1; v2 <- s2^2 / n2
    se <- sqrt(v1 + v2); df <- (v1 + v2)^2 / (v1^2 / (n1 - 1) + v2^2 / (n2 - 1))
  }
  d <- m1 - m2; t <- d / se; crit <- qt(0.975, df)
  round(c(diff = d, se = se, t = t, df = df, p = 2 * pt(-abs(t), df),
          lower = d - crit * se, upper = d + crit * se), 4)
}
t_from_stats(12.4, 7.0, 30, 8.1, 7.2, 30, var.equal = TRUE)   # 0장의 혈압 시험

### 셀 8. Welch와 Student가 다른 답을 낼 때

In [ ]:
t_from_stats(-0.9, 0.8, 36, -0.4, 0.9, 36)                     # 2장 복약상담, Welch
rbind(Student = t_from_stats(-0.90, 0.60, 40, -0.40, 1.20, 15, var.equal = TRUE),
      Welch   = t_from_stats(-0.90, 0.60, 40, -0.40, 1.20, 15))   # 2장 예비연구

### 셀 9. 가상 논문의 숫자 검산 (분석 고르기 연습 d 절)

In [ ]:
t_from_stats(84.9, 15.1, 150, 79.1, 19.3, 150)                    # 평균 PDC: Welch (논문의 선택)
t_from_stats(84.9, 15.1, 150, 79.1, 19.3, 150, var.equal = TRUE)  # Student로 바꾸면
tab <- matrix(c(108, 42,
                90, 60), nrow = 2, byrow = TRUE)                  # PDC ≥ 80%: 순응, 비순응 (인원)
unlist(chisq.test(tab, correct = FALSE)[c("statistic", "p.value")])   # 연속성 보정 없이 (논문의 선택)
unlist(chisq.test(tab)[c("statistic", "p.value")])                    # R 기본값: Yates 보정

## d. 같은 표의 χ²와 z, 회귀 출력의 z

### 셀 10. 같은 2 × 2 표의 χ²와 z

In [ ]:
tab <- matrix(c(12, 38,
                18, 32), nrow = 2, byrow = TRUE,
              dimnames = list(group = c("treat", "control"), event = c("yes", "no")))
chisq.test(tab)$statistic                    # 기본값: Yates 보정
chisq.test(tab, correct = FALSE)$p.value
pt2 <- prop.test(c(12, 18), c(50, 50), correct = FALSE)
c(X2 = unname(pt2$statistic), z = -sqrt(unname(pt2$statistic)), p = pt2$p.value)

### 셀 11. z, Wald χ², 오즈비를 계수와 표준오차로

In [ ]:
b <- 0.6420; se <- 0.2110          # 로지스틱 회귀 출력의 계수와 표준오차
z <- b / se
round(c(z = z, wald = z^2), 3)
round(c(p_z = 2 * pnorm(-abs(z)), p_chisq = pchisq(z^2, 1, lower.tail = FALSE)), 4)
round(exp(c(OR = b, lower = b - 1.96 * se, upper = b + 1.96 * se)), 2)

## e. 다중비교 보정과 부트스트랩

### 셀 12. 다중비교 보정

In [ ]:
p <- c(0.004, 0.012, 0.021, 0.038, 0.260)
rbind(holm = p.adjust(p),                    # 기본값이 Holm
      BH = p.adjust(p, method = "BH"),
      bonferroni = p.adjust(p, method = "bonferroni"))

### 셀 13. 평균 재원일수의 부트스트랩 신뢰구간

In [ ]:
d <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/explore60.csv")       # 1장 h 절의 60명
set.seed(2026)
m <- replicate(2000, mean(sample(d$los, replace = TRUE)))
round(quantile(m, c(0.025, 0.975)), 2)      # 백분위수 부트스트랩 95% CI
library(boot)
bt <- boot(d$los, function(x, i) mean(x[i]), R = 2000)
boot.ci(bt, type = c("perc", "bca"))

## f. 분위수, 왜도, 정규성

### 셀 14. 분위수와 왜도는 정의에 따라 다르다

In [ ]:
quantile(d$los, c(0.25, 0.75))                 # 기본값 type = 7
quantile(d$los, c(0.25, 0.75), type = 6)       # SPSS의 기본값과 같음
library(e1071)
round(c(type1 = skewness(d$los, type = 1),      # 표본 그대로(scipy 기본값)
        type2 = skewness(d$los, type = 2),      # 표본 크기 보정(SPSS, pandas)
        type3 = skewness(d$los)), 2)            # e1071의 기본값은 type 3

### 셀 15. Q-Q plot

In [ ]:
par(mfrow = c(1, 2))
qqnorm(d$sbp, main = "Systolic BP"); qqline(d$sbp)
qqnorm(d$los, main = "Length of stay"); qqline(d$los)

### 셀 16. 정규성 검정을 쓸 때의 함정

In [ ]:
shapiro.test(d$los)$p.value
ks.test(d$sbp, "pnorm")$p.value            # 표준정규분포와 비교해 버림
ks.test(as.numeric(scale(d$los)), "pnorm")$p.value   # 같은 자료의 평균·SD로 표준화
library(nortest)
lillie.test(d$los)$p.value                 # 이 점을 보정한 Lilliefors 검정

## g. 순위합 검정의 세부

### 셀 17. 같은 자료에서 p를 구하는 세 방법

In [ ]:
A <- c(1, 2, 3, 3, 4, 5, 47); B <- 6:12       # 1장 g 절의 재원일수 7명씩
wilcox.test(A, B, exact = FALSE)$p.value                    # 정규근사(동점 보정, 연속성 보정)
wilcox.test(A, B, exact = FALSE, correct = FALSE)$p.value   # 연속성 보정 없이
suppressPackageStartupMessages(library(coin))
g <- factor(rep(c("A", "B"), each = 7))
pvalue(wilcox_test(c(A, B) ~ g, distribution = "exact"))   # 중간순위 그대로 정확 p

### 셀 18. 오피오이드 자료의 Wilcoxon 순위합 검정과 Hodges–Lehmann 추정치

In [ ]:
protocol <- c(12, 15, 18, 20, 22, 26, 31, 68)   # 프로토콜군 8명
usual    <- c(24, 29, 36, 41, 45, 58, 115)      # 일반 관리군 7명
wilcox.test(protocol, usual, conf.int = TRUE)

### 셀 19. 동점이 많은 리커트 점수

In [ ]:
counsel <- rep(1:5, times = c(0, 2, 5, 8, 5))   # 상담군 만족도 20명
control <- rep(1:5, times = c(2, 5, 7, 4, 2))   # 대조군 20명
wilcox.test(counsel, control, exact = FALSE)$p.value     # 동점 보정 + 연속성 보정
wilcox.test(counsel, control, exact = FALSE, correct = FALSE)$p.value
set.seed(2026)
g2 <- factor(rep(c("counsel", "control"), each = 20))
pvalue(wilcox_test(c(counsel, control) ~ g2,
                   distribution = approximate(nresample = 10000)))

### 셀 20. 퍼진 정도가 다를 때: Brunner–Munzel 검정

In [ ]:
library(brunnermunzel)
brunnermunzel.test(protocol, usual)

## h. 과제 정답

### 셀 21. 과제 1 정답: 결측이 있는 열의 요약

In [ ]:
y <- c(132, NA, 128, 141, NA, 137, 125)
c(n_used = sum(!is.na(y)), mean = mean(y, na.rm = TRUE), sd = round(sd(y, na.rm = TRUE), 2))

### 셀 22. 과제 2 정답: 로그 변환한 재원일수

In [ ]:
round(c(p_raw = shapiro.test(d$los)$p.value, p_log = shapiro.test(log(d$los))$p.value), 4)
round(exp(mean(log(d$los))), 2)            # 기하평균

### 셀 23. 과제 3 정답: 큰 군의 SD가 클 때

In [ ]:
t_from_stats(-0.90, 1.20, 40, -0.40, 0.60, 15, var.equal = TRUE)["p"]
t_from_stats(-0.90, 1.20, 40, -0.40, 0.60, 15)["p"]